# Part 1 — Scope, Ownership, and Outputs

### Objective

Build the local analysis-ready dataset for dynamic AKI prediction. This notebook owns source validation, cohort construction, KDIGO onset timelines, hourly snapshots, multi-horizon labels, leakage-safe features, and patient-level splits.

### Outputs

- Eligible ICU cohort
- Dataset-local patient/stay identifier crosswalk
- Hourly snapshot dataset
- Patient-level split manifest
- Feature dictionary and aggregate quality report

### Rules

Do not train models here, upload MIMIC data, or commit patient-level artifacts.


In [ ]:
# `eligible_cohort` must contain exactly one selected ICU stay per patient.
# Keep source IDs for provenance; these local IDs are not model features.
required = {'subject_id', 'hadm_id', 'icustay_id'}
missing = required.difference(eligible_cohort.columns)
if missing:
    raise ValueError(f"eligible_cohort is missing required columns: {sorted(missing)}")
if eligible_cohort[list(required)].isna().any().any():
    raise ValueError("subject_id, hadm_id, and icustay_id must not be missing")

cohort = eligible_cohort.copy()
cohort = cohort.sort_values(['subject_id', 'icustay_id'], kind='mergesort').reset_index(drop=True)
if cohort['subject_id'].duplicated().any():
    raise ValueError("Cohort contains multiple selected ICU stays for one subject_id")

cohort['patient_id'] = [f'patient_{i:03d}' for i in range(1, len(cohort) + 1)]
cohort['stay_id'] = [f'stay_{i:03d}' for i in range(1, len(cohort) + 1)]
id_crosswalk = cohort[['patient_id', 'stay_id', 'subject_id', 'hadm_id', 'icustay_id']].copy()
assert id_crosswalk['patient_id'].is_unique
assert id_crosswalk['stay_id'].is_unique

# `cohort` and `id_crosswalk` are passed to the split-manifest step.


# Part 2 — Configuration and Study Definitions

### Objective

Load all study choices from configuration before processing data.

### Required settings

- Local MIMIC-III and artifact roots
- Monitoring start and end times
- Training snapshot interval: 1 hour
- Prediction horizons: 6, 12, 24, and 48 hours
- Primary target: creatinine-based AKI only; urine-output AKI is an alternative robustness endpoint
- Operational horizon: selected on validation data
- Feature lookback windows
- Follow-up and exclusion rules
- Minimum follow-up duration and minimum observation/measurement coverage rules
- Patient split proportions and random seed
- AKI and baseline-creatinine definition versions
- Urine-output KDIGO standard/version, weight source, rolling windows, missing-output handling, and renal replacement therapy rules

### Rule

Do not redefine these choices in later cells without creating a new dataset version.


# Part 3 — Load and Validate MIMIC-III Tables

### Objective

Load only required columns and standardize identifiers, timestamps, units, and missing values.

### Candidate sources

PATIENTS, ADMISSIONS, ICUSTAYS, LABEVENTS, CHARTEVENTS, OUTPUTEVENTS, and approved diagnosis, procedure, medication, and intervention tables. OUTPUTEVENTS and the prespecified weight source are required if urine-output KDIGO is enabled.

### Checks

- subject_id, hadm_id, and icustay_id relationships
- Source counts and timestamp ranges
- CareVue and MetaVision item mappings
- Duplicate, invalid, and incompatible-unit measurements

### Processing requirement

Process large event tables such as LABEVENTS, CHARTEVENTS, and OUTPUTEVENTS in chunks or stay-level partitions; do not load the full table into memory. Preserve identifiers, timestamps, units, and provenance within every chunk before aggregation.

### Output

Validated local source views with explicit schemas and source-ID relationships; the final dataset-local ID crosswalk is created after cohort selection in Part 4.


# Part 4 — Define the Base ICU Cohort

### Objective

Create the eligible population before repeated snapshots are generated.

### Inclusion rules

- Adults under the prespecified MIMIC age rule
- Exactly one selected ICU stay per patient, chosen by a prespecified eligibility/chronology rule
- Retain patients with sufficient follow-up for at least one prediction horizon; determine eligibility separately for every snapshot and horizon.

### Exclusion rules

- ESKD or chronic dialysis under prespecified definitions
- Invalid ICU timing
- Additional exclusions documented before use

### Dataset-local identifiers

After the eligible cohort is finalized, create a deterministic `patient_id` for each included `subject_id` and a deterministic `stay_id` for each selected `icustay_id`. Store the one-to-one crosswalk with the cohort and never use these IDs to make clinical features.

### Output

One row per eligible patient and selected ICU stay, including patient_id, stay_id, subject_id, hadm_id, icustay_id, and the selection-rule version.


# Part 5 — Define Baseline Creatinine and Renal History

### Objective

Create a chronological creatinine record and assign the primary baseline used for KDIGO assessment.

### Rules

- Standardize units and preserve measurement provenance.
- Define how pre-ICU, admission, nadir, and missing baselines are handled.
- Do not use future measurements to construct a snapshot's input features.
- Record CKD, prior renal replacement therapy, and exclusion evidence separately.
- Prepare urine-output records with units, timestamps, source provenance, and the weight required for mL/kg/h calculations when the urine-output robustness label is enabled.

### Sensitivity plan

Prepare alternative defensible baseline definitions for later robustness analysis rather than silently choosing the most favorable result.


# Part 6 — Generate KDIGO AKI Onset Timelines

### Objective

Identify the earliest AKI onset, stage, and supporting criterion.

### Primary definition

Creatinine increase of at least 0.3 mg/dL within 48 hours or at least 1.5 times the applicable baseline within the prespecified KDIGO period.

Use creatinine-based onset for the primary AKI task. Compute urine-output onset separately as an alternative robustness endpoint; do not let urine output silently replace the primary onset. Do not construct a combined-label training target in this notebook set.

### Robustness extension

Add urine-output KDIGO 2012 using the configured documented body-weight source, rolling windows, missing-output handling, and renal replacement therapy rules. Record the standard/version and all rule choices in dataset provenance.

### Checks

Use manually constructed positive, negative, boundary, duplicate-time, missing-baseline, and low-urine-output cases.


# Part 7 — Freeze Patient Identity and Development Splits

### Objective

Use the finalized one-patient/one-selected-stay cohort from Part 4 and assign each patient to exactly one reproducible development split before generating any snapshots, labels, or features.

### Rules

- Use the dataset-local `patient_id` and `stay_id` created in Part 4; retain `subject_id`, `hadm_id`, and `icustay_id` as provenance keys.
- Write one split-manifest row per patient/selected stay.
- Assign all future rows for that patient to the inherited train, validation, or test split.
- Stop before proceeding if any patient has multiple selected stays, a missing split, or a split overlap.

### Output

One row per selected patient/stay in the frozen cohort and split manifest.


# Part 8 — Generate Hourly Training Snapshots

### Objective

Create one candidate prediction cutoff per selected ICU stay and ICU hour during the configured monitoring period.

### Rules

- Inherit `patient_id`, `subject_id`, `hadm_id`, `icustay_id`, and the preassigned split from the frozen manifest.
- feature_time <= snapshot_time.
- Stop snapshots at AKI onset, ICU discharge, death, or monitoring end; no snapshot at or after AKI onset may enter the final modeling table.
- Record ICU discharge/death/censoring time, available follow-up duration, and observation/measurement coverage for each snapshot.
- Keep actual AKI-relevant EHR event times for later event-driven replay.
- Do not create minute-level duplicate training rows when no relevant data changed.

### Output

One candidate row per selected `icustay_id` and `snapshot_time`, with inherited patient and split keys.


# Part 9 — Assign 6h, 12h, 24h, and 48h AKI Targets

### Objective

For every snapshot, assign horizon-specific eligibility and labels.

### Label rule

For horizon H, the positive window is (snapshot_time, snapshot_time + H]. A patient with AKI at or before snapshot_time is not at risk and must not contribute a later snapshot.

A horizon is eligible only when the outcome is observable through snapshot_time + H, or an AKI event is observed before that time. If ICU discharge, death, or another censoring boundary occurs first without AKI, set the horizon to ineligible rather than negative.
The configured minimum observation/measurement coverage rules must be reported separately from horizon eligibility; insufficient data coverage must not be silently treated as a negative outcome.

### Required fields

- eligible_6h, aki_within_6h
- eligible_12h, aki_within_12h
- eligible_24h, aki_within_24h
- eligible_48h, aki_within_48h
- creatinine_aki_onset_time when applicable
- urine_output_aki_onset_time and alternative robustness labels when enabled
- icu_discharge_time, censoring_time, followup_hours, observation_coverage

Overlapping labels across hours and horizons are expected and are not leakage.

### Next gate

Labels are created after the cohort and split are frozen and before feature extraction.


# Part 10 — Extract Leakage-Safe Multimodal Features

### Objective

Summarize information available by each snapshot.

### Modalities

- Demographics and comorbidities
- Vital signs and laboratory measurements
- Glasgow Coma Scale
- Urine output
- Medications and interventions when feasible

### Summaries

Use prespecified latest, mean, minimum, maximum, range, slope, variability, count, missingness, and time-since-last-measurement features over clinically meaningful windows.

### Caution

Measurement frequency can encode workflow and illness severity; retain it only when intentional and test its transportability.


# Part 11 — Build the Dynamic Modeling Table

### Objective

Join all snapshot-specific features and targets into a stable schema while preserving the frozen patient/stay identity and split assignment.

### Required fields

- patient_id, subject_id, hadm_id, and icustay_id
- split, snapshot_time, and continuous hours_since_icu
- Horizon-specific labels and eligibility flags
- Feature columns and provenance version
- CareVue or MetaVision source indicator for robustness analysis

### Rules

Keep identifiers, outcomes, onset times, split columns, and future information out of the model feature list. Preserve 6h, 12h, 24h, and 48h as horizon-specific targets.

### Temporal generalization note

Every snapshot from a patient must inherit the same split; never re-split this table by row. MIMIC-III calendar years are patient-shifted and must not be treated as true chronology.


# Part 12 — Validate and Export Dataset Artifacts

### Objective

Run final integrity checks and write versioned local artifacts.

### Required checks

- One-to-one patient_id to subject_id mapping
- One selected icustay_id per patient_id
- Patient disjointness across splits
- No feature event after its snapshot
- No snapshot at or after AKI onset in the final modeling table
- Configured minimum follow-up and observation/measurement coverage rules are applied and reported
- Correct horizon boundaries and follow-up eligibility
- Unique snapshot keys
- Outcome prevalence and sample counts by split, hour, horizon, ICU type, and source system
- Reconciliation of horizon-specific outcome prevalence and known cohort counts

### Outputs

Export the cohort, snapshot dataset, split manifest, feature dictionary, configuration, and aggregate quality report. Never commit patient-level outputs.
